# BanglaSQL — Colab Training Notebook
**Natural Language (Bangla) to SQL — University Management System**

### Steps
Run each cell **in order**. Runtime → Change runtime type → **T4 GPU** before starting.

#Connect Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — Check GPU

In [2]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

GPU available: True
GPU name: Tesla T4
VRAM: 15.6 GB


## Step 2 — Clone repository

In [3]:
# Replace with your actual GitHub repo URL
REPO_URL = 'https://github.com/mustafiz-07/BanglaSQL.git'

!git clone {REPO_URL} banglasql
%cd banglasql/
!ls -la

Cloning into 'banglasql'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 88 (delta 48), reused 64 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 111.15 KiB | 18.52 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/banglasql
total 308
drwxr-xr-x 4 root root   4096 Sep 13 17:27 .
drwxr-xr-x 1 root root   4096 Sep 13 17:27 ..
-rw-r--r-- 1 root root   5396 Sep 13 17:27 app.py
-rw-r--r-- 1 root root  16836 Sep 13 17:27 build_dataset.py
-rw-r--r-- 1 root root 175577 Sep 13 17:27 colab_train.ipynb
-rw-r--r-- 1 root root   2388 Sep 13 17:27 common.py
-rw-r--r-- 1 root root  14853 Sep 13 17:27 create_database.py
drwxr-xr-x 2 root root   4096 Sep 13 17:27 data
-rw-r--r-- 1 root root    232 Sep 13 17:27 docker-compose.yml
-rw-r--r-- 1 root root    360 Sep 13 17:27 Dockerfile
-rw-r--r-- 1 root root   2331 Sep 13 17:27 er_diagram.md
-rw-r--r-- 1 root root  12757 Sep 

## Step 3 — Install dependencies

In [4]:
# NOTE: train.py's own Quick Start instructions call for
# requirements_colab.txt (a Colab-specific pin list — Colab already ships a
# working torch/CUDA build, so this avoids reinstalling a conflicting one).
# This cell previously referenced a plain 'requirements.txt', which doesn't
# match and would fail with 'file not found' on a fresh clone.
!pip install -r requirements_colab.txt -q
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 85.7 MB/s eta 0:00:00
Dependencies installed.


## Step 4 — Generate database & dataset

In [5]:
!python create_database.py

Creating schema in: /content/banglasql/banglasql.db
Populating with synthetic data...

BanglaSQL Database — Summary
  departments     :    10 rows
  instructors     :    50 rows
  students        :   304 rows
  courses         :    80 rows
  enrollments     :   471 rows
  attendance      :   500 rows

-- Sample: Top 5 students by CGPA --
  Tasnim Sultana — CGPA: 4.0
  Farhana Rahman — CGPA: 3.97
  Nazmul Alam — CGPA: 3.96
  Joynal Mondal — CGPA: 3.96
  Tasnim Begum — CGPA: 3.96

-- Sample: Students per department --
  Mechanical Engineering                        : 34 students
  Physics                                       : 34 students
  Computer Science and Engineering              : 33 students
  Business Administration                       : 32 students
  Civil Engineering                             : 32 students
  Mathematics                                   : 30 students
  Electrical and Electronic Engineering         : 29 students
  English                                   

In [6]:
!python build_dataset.py

BanglaSQL Dataset Builder — Phase 2

[1/5] Loaded 165 base templates
      Easy  : 65
      Medium: 100

[2/5] After augmentation: 990 pairs

[3/5] After deduplication: 990 pairs (6.0x base)

[4/5] Split (by template, stratified by query_type):
      Train :  642 (65%)
      Dev   :  168 (17%)
      Test  :  180 (18%)

  [OK] All 28 query types present in train.
  [OK] Test SQL queries also seen in train: 0 (0 = no leakage)

[5/5] Saved /content/banglasql/data/dataset_train.json

[5/5] Saved /content/banglasql/data/dataset_dev.json

[5/5] Saved /content/banglasql/data/dataset_test.json

      Saved stats: /content/banglasql/data/dataset_stats.json

{
  "base_templates": {
    "easy": 65,
    "medium": 100,
    "total": 165
  },
  "after_augmentation_dedup": {
    "easy": 390,
    "medium": 600,
    "total": 990,
    "augmentation_ratio": 6.0
  },
  "splits": {
    "train": 642,
    "dev": 168,
    "test": 180
  },
  "templates_per_split": {
    "train": 107,
    "dev": 28,
    "test": 

## Step 5 — Tokenizer analysis & preprocessing

In [7]:
!python preprocess_check.py

BanglaSQL — Phase 3: Preprocessing & Tokenizer Analysis

[1/3] Unicode Normalization (NFC)...
  train: 642 pairs, 0 normalized
  dev: 168 pairs, 0 normalized
  test: 180 pairs, 0 normalized

  (Analysis below uses train+dev only — 810 pairs. Test split (180 pairs) is excluded from here on so hyperparameter choices can't leak information from it.)

[2/3] Tokenizer Analysis...

  Loading tokenizer: csebuetnlp/banglat5
config.json: 100% 659/659 [00:00<00:00, 1.95MB/s]
tokenizer_config.json: 100% 1.83k/1.83k [00:00<00:00, 5.66MB/s]

spiece.model: downloading bytes:   0% 0.00/1.11M [00:00<?, ?B/s]
spiece.model: downloading bytes: 100% 697k/697k [00:01<00:00, 537kB/s, 67.3kB/s  ]
spiece.model: reconstructing file: 100% 1.11M/1.11M [00:01<00:00, 857kB/s,  107kB/s  ]
special_tokens_map.json: 100% 1.79k/1.79k [00:00<00:00, 5.57MB/s]

  Model: csebuetnlp/banglat5
  Vocab size: 32,100
  Bangla-script tokens in vocab: 28,644 (89.2%)
  Avg tokens per question (sample of 50): 12.3
  Avg fragment sub

## Step 6 — Train
> Up to 25 epochs with early stopping (patience 5, 6-epoch warm-up).
> The best checkpoint is chosen on dev **execution accuracy** (greedy decoding during
> training) and saved to `checkpoints/best_model` together with `banglasql_config.json`,
> which records the exact input format the model was trained on.
> Epochs are much faster than the first run: the model input is now the question only,
> not question + a ~220-token schema string.

In [8]:
!python train.py

2026-09-13 17:29:05,096 [INFO] ============================================================
2026-09-13 17:29:05,096 [INFO] BanglaSQL — Phase 3: Training
2026-09-13 17:29:05,096 [INFO] ============================================================
2026-09-13 17:29:05,096 [INFO] Model      : csebuetnlp/banglat5
2026-09-13 17:29:05,096 [INFO] Max in/out : 256/128
2026-09-13 17:29:05,096 [INFO] Batch      : 8 x1 accum
2026-09-13 17:29:05,096 [INFO] Epochs     : 40   LR: 0.0003
2026-09-13 17:29:05,096 [INFO] Early stop : patience=6, warm-up=8
2026-09-13 17:29:05,096 [INFO] Device     : cuda
2026-09-13 17:29:05,124 [INFO] GPU        : Tesla T4
2026-09-13 17:29:05,125 [INFO] 
Loading tokenizer & model: csebuetnlp/banglat5
2026-09-13 17:29:05,435 [INFO] HTTP Request: HEAD https://huggingface.co/csebuetnlp/banglat5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-13 17:29:05,435 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to e

## Step 7 — Save model to Google Drive (prevents loss on session timeout)

In [9]:


import shutil, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/BanglaSQL/checkpoints3'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy the best model checkpoint
src = 'checkpoints/best_model'
dst = os.path.join(DRIVE_SAVE_DIR, 'best_model')

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Best model saved to Google Drive: {dst}')
else:
    print('No best_model found. Check that training completed successfully.')

Best model saved to Google Drive: /content/drive/MyDrive/BanglaSQL/checkpoints3/best_model


## Step 8 — Quick inference test (verify the model works)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from common import format_input, load_config

MODEL_DIR = 'checkpoints/best_model'
cfg       = load_config(MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
model.eval()

def predict(bangla_question: str) -> str:
    inp = format_input(bangla_question, cfg)
    ids = tokenizer(inp, return_tensors='pt',
                    max_length=cfg['max_input_length'], truncation=True)
    out = model.generate(**ids, max_length=cfg['max_target_length'],
                         num_beams=4, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_questions = [
    'সকল শিক্ষার্থীর তালিকা দাও।',
    'যেসব শিক্ষার্থীর CGPA ৩.৫-এর বেশি তাদের নাম দাও।',
    'প্রতিটি বিভাগে কতজন শিক্ষার্থী আছে তা দেখাও।',
]

print('=== Inference Test ===\n')
for q in test_questions:
    print(f'Q  : {q}')
    print(f'SQL: {predict(q)}\n')

## Step 9 — Evaluate on the test split (Phase 4)
Decodes with beam search (4 beams) and reports two rows side by side:
- **top-1 beam** — the model's highest-scoring query
- **execution-guided** — the highest-scoring beam that actually executes (Wang et al., 2018)

Metrics: execution accuracy, exact match, validity rate, per-component accuracy and a
failure-category breakdown. Results land in `logs/test_results.json`, per-example
predictions in `logs/test_predictions.json`.

In [11]:
!python evaluate.py --split test

BanglaSQL — Phase 4: Evaluation (test split, 180 examples)
Checkpoint: /content/banglasql/checkpoints/best_model
Loading weights: 100% 282/282 [00:00<00:00, 5143.10it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.

Generating SQL...
  generated 180/180
Executing queries against the database...

RESULTS — test split (180 examples)
  Execution accuracy : 27.2%
  Exact match        : 21.7%
  Validity rate      : 63.3%

  By difficulty:
    easy     n=  72  exec=33.3%  em=29.2%
    medium   n= 108  exec=23.2%  em=16.7%

  Component accuracy:
    aggregate  n=  71  53.5%
    group_by   n=  59  57.6%
    having     n=  14  7.1%
    join       n=  97  57.7%
    order_by   n=  42  28.6%
    select     n= 180  43.3%
    tables     n= 180  71.1%
    whe

In [ ]:
import json

with open('logs/test_results.json') as f:
    results = json.load(f)

top1, guided = results['top1_metrics'], results['metrics']
print(f"{'':20s} {'top-1 beam':>12s} {'exec-guided':>12s}")
for key, label in [('execution_accuracy', 'Execution accuracy'),
                   ('exact_match', 'Exact match'),
                   ('validity_rate', 'Validity rate')]:
    print(f'{label:20s} {top1[key]:>12.1%} {guided[key]:>12.1%}')

print('\nBy difficulty (exec-guided):')
for tier, s in results['by_difficulty'].items():
    print(f"  {tier:8s} n={s['n']:4d}  exec={s['execution_accuracy']:.1%}")

print('\nFailure categories (exec-guided):')
for name, count in results['failure_categories'].items():
    print(f'  {name:22s} {count}')

### Training curves (for the report)
Plots loss and dev exact-match per epoch from `logs/train_history.json`.

In [ ]:
import json
import matplotlib.pyplot as plt

with open('logs/train_history.json') as f:
    history = json.load(f)

def series(key):
    return [(h['epoch'], h[key]) for h in history if key in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(*zip(*series('loss')), label='train loss')
ax1.plot(*zip(*series('eval_loss')), label='dev loss')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.set_title('Loss')

ax2.plot(*zip(*series('eval_execution_accuracy')), label='execution accuracy')
ax2.plot(*zip(*series('eval_exact_match')), label='exact match')
ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend(); ax2.set_title('Dev accuracy (greedy)')
plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150)
plt.show()

## Step 10 — Download model
If you prefer downloading the checkpoint directly instead of using Drive:

In [14]:
import shutil
shutil.make_archive('best_model', 'zip', 'checkpoints/best_model')

from google.colab import files
files.download('best_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>